In [1]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (10).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [3]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [20]:
df["Кейс"].value_counts()

,count
Кейс,
1,454
8,128
2,121
4,120
6,120
5,120
10,120
9,120
3,118


In [33]:
df["Доказательства"].value_counts()

,count
Доказательства,
4,394
2,323
3,320
5,289
1,206
6,1


In [ ]:
df["Оценка"].value_counts()

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


In [6]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [7]:
class BertClassifier(nn.Module):
  def __init__(self, num_classes, model_name='bert-base-multilingual-cased'):
    super(BertClassifier, self).__init__()
    self.bert = BertModel.from_pretrained(model_name)
    for param in self.bert.parameters():
      param.requires_grad=False
    self.classifier = nn.Sequential(nn.Dropout(0.3),
                                    nn.Linear(self.bert.config.hidden_size, 256),
                                    nn.ReLU(),
                                    nn.Dropout(0.2),
                                    nn.Linear(256, num_classes)
    )
  def forward(self, input_ids, attention_mask):
    with torch.no_grad():
      outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
      cls = outputs.last_hidden_state[:, 0, :]
    logits = self.classifier(cls)
    return logits

In [8]:
tokenizer= BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertClassifier(num_classes=5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def train(loader):
    total_loss = 0
    model.train()
    for batch in loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      optimizer.zero_grad()
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    return total_loss / len(loader)


In [10]:
def evals(loader):
    model.eval()
    loss_lst = []
    predictions = []
    total = 0
    correct = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

            loss_lst.append(loss.item())
            correct += (preds == labels).sum().item()
            total += len(labels)
            predictions.extend(preds.cpu().numpy())

    return np.mean(loss_lst), correct / total, np.array(predictions)

In [11]:
def predicts(texts, model, tokenizer, device, label_encoder=None, max_len=128):
  model.eval()
  predictions = []
  with torch.no_grad():
    for text in texts:
      encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
      input_ids = encoding["input_ids"].to(device)
      attention_mask = encoding["attention_mask"].to(device)
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      probs = torch.softmax(outputs, dim=1)
      pred = torch.argmax(probs, dim=1)
      predictions.append(pred.cpu().numpy()[0])
  return np.array(predictions)

In [28]:
X_text_audience = df['Решение кейса'].fillna('').values
y_audience = (df['ЦА'].values - 1).astype("int64")

X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text_audience, y_audience, test_size=0.2, random_state=42, stratify=y_audience
)
X_text_dataset_audience = TextDataset(X_train_text_audience, y_train_audience, tokenizer)
X_val_dataset_audience = TextDataset(X_test_text_audience, y_test_audience, tokenizer)
X_text_dataloader_audience = DataLoader(X_text_dataset_audience, batch_size=16, shuffle=True)
X_val_dataloader_audience = DataLoader(X_val_dataset_audience, batch_size=16, shuffle=False)
model_audience = BertClassifier(num_classes=5)
model_audience.to(device)
optimizer_audience = torch.optim.AdamW(model_audience.classifier.parameters(), lr=2e-4)
criterion_audience = nn.CrossEntropyLoss()
model = model_audience
model.to(device)
optimizer = optimizer_audience
criterion = criterion_audience


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
best_loss_audience = float("inf")
loss_all = 0
loss_val = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_audience)
  loss_all += train_loss
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_audience)
  loss_val+=val_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_audience:
    best_loss_audience = val_loss
    torch.save(model_audience.state_dict(), "best_model_audience.pt")
print("best_val_loss", best_loss_audience)
print("avg_train", loss_all / 10)
print("avg_val", loss_val / 10)

train_loss= 1.5607814649482825
val_loss= 1.5268710255622864
val_acc= 0.2703583061889251
train_loss= 1.5295253233476118
val_loss= 1.505339354276657
val_acc= 0.28664495114006516
train_loss= 1.494301351633939
val_loss= 1.477669095993042
val_acc= 0.2996742671009772
train_loss= 1.4699685743876867
val_loss= 1.4561225771903992
val_acc= 0.3127035830618892
train_loss= 1.4561942521627846
val_loss= 1.4342579305171967
val_acc= 0.31921824104234525
train_loss= 1.4315289646000058
val_loss= 1.4219938695430756
val_acc= 0.3485342019543974
train_loss= 1.405564633282748
val_loss= 1.4096236050128936
val_acc= 0.3517915309446254
train_loss= 1.3930044793463372
val_loss= 1.3995451092720033
val_acc= 0.3583061889250814
train_loss= 1.3822370690184753
val_loss= 1.378831422328949
val_acc= 0.34527687296416937
train_loss= 1.3644154427887558
val_loss= 1.3675637125968934
val_acc= 0.3517915309446254
best_val_loss 1.3675637125968934
avg_train 1.4487521555516627
avg_val 1.4377817702293396


In [30]:
model_audience.load_state_dict(torch.load("best_model_audience.pt"))
model_audience.to(device)
model_audience.eval()
_, _, predictions = evals(X_val_dataloader_audience)
true_labels = []
for batch in X_val_dataloader_audience:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.70      0.25      0.37        28
           1       0.38      0.36      0.37        55
           2       0.18      0.04      0.07        70
           3       0.33      0.80      0.46        83
           4       0.46      0.17      0.25        71

    accuracy                           0.35       307
   macro avg       0.41      0.32      0.30       307
weighted avg       0.37      0.35      0.30       307

MAE 0.8599348534201955


In [31]:
_, _, predictions_audience = evals(X_text_dataloader_audience)
true_labels_audience = []
for batch in X_text_dataloader_audience:
  labels = batch["labels"].cpu().numpy()
  true_labels_audience.extend(labels)
print(classification_report(true_labels_audience, predictions_audience))
print("MAE", mean_absolute_error(true_labels_audience, predictions_audience))

              precision    recall  f1-score   support

           0       0.15      0.08      0.10       114
           1       0.15      0.16      0.16       222
           2       0.28      0.09      0.14       279
           3       0.27      0.57      0.36       329
           4       0.21      0.11      0.14       282

    accuracy                           0.23      1226
   macro avg       0.21      0.20      0.18      1226
weighted avg       0.23      0.23      0.20      1226

MAE 1.3140293637846656


In [32]:
X_text_sol = df['Решение кейса'].fillna('').values
y_sol = (df['Проработка решения'].values - 1).astype("int64")

X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text_sol, y_sol, test_size=0.2, random_state=42, stratify=y_sol
)
X_text_dataset_sol = TextDataset(X_train_text_sol, y_train_sol, tokenizer)
X_val_dataset_sol = TextDataset(X_test_text_sol, y_test_sol, tokenizer)
X_text_dataloader_sol = DataLoader(X_text_dataset_sol, batch_size=16, shuffle=True)
X_val_dataloader_sol = DataLoader(X_val_dataset_sol, batch_size=16, shuffle=False)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [26]:
model_sol = BertClassifier(num_classes=5)
model_sol.to(device)
optimizer_sol = torch.optim.AdamW(model_sol.classifier.parameters(), lr=2e-4)
criterion_sol = nn.CrossEntropyLoss()
model = model_sol
model.to(device)
optimizer = optimizer_sol
criterion = criterion_sol


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
y_sol.value_counts()

AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

In [28]:
best_loss_sol = float("inf")
train_all = 0
val_all = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_sol)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_sol)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  train_all += train_loss
  val_all+= val_loss
  if val_loss <= best_loss_sol:
    best_loss_sol = val_loss
    torch.save(model_sol.state_dict(), "best_model_sol.pt")
print("avg_train=,", train_all/10)
print("avg_val=,", val_all/10)



train_loss= 1.57464925190071


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
model_sol.load_state_dict(torch.load("best_model_sol.pt"))
model_sol.to(device)
model_sol.eval()
_, _, predictions = evals(X_text_dataloader_sol)
true_labels = []
for batch in X_val_dataloader_sol:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.50      0.42      0.45        12
           1       0.00      0.00      0.00        16
           2       0.53      0.72      0.61        29
           3       0.44      0.41      0.42        27
           4       0.44      0.69      0.54        16

    accuracy                           0.48       100
   macro avg       0.38      0.45      0.40       100
weighted avg       0.40      0.48      0.43       100

MAE 0.82


In [ ]:
_, _, predictions_train = evals(X_val_dataloader_sol)
true_labels_train = []
for batch in X_text_dataloader_sol:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

In [33]:
model_finance = BertClassifier(num_classes=5)
model_finance.to(device)
optimizer_finance = torch.optim.AdamW(model_finance.classifier.parameters(), lr=2e-4)
criterion_finance = nn.CrossEntropyLoss()
model = model_finance
model.to(device)
optimizer = optimizer_finance
criterion = criterion_finance


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
X_text_finance = df['Решение кейса'].fillna('').values
y_finance = (df['Финансовая модель и метрики'].values - 1).astype("int64")

X_train_text_finance, X_test_text_finance, y_train_finance, y_test_finance = train_test_split(
    X_text_finance, y_finance, test_size=0.2, random_state=42,stratify= y_finance
)
X_text_dataset_finance = TextDataset(X_train_text_finance, y_train_finance, tokenizer)
X_val_dataset_finance = TextDataset(X_test_text_finance, y_test_finance, tokenizer)
X_text_dataloader_finance = DataLoader(X_text_dataset_finance, batch_size=16, shuffle=True)
X_val_dataloader_finance = DataLoader(X_val_dataset_finance, batch_size=16, shuffle=False)

In [35]:
best_loss_finance = float("inf")
all_train = 0
all_val= 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_finance)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_finance)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  all_train += train_loss
  all_val += val_loss
  if val_loss <= best_loss_finance:
    best_loss_finance = val_loss
    torch.save(model_finance.state_dict(), "best_model_finance.pt")
print("avg_val", all_val / 10)
print("avg_train", all_train/10)

train_loss= 1.5663615604499719
val_loss= 1.531438845396042
val_acc= 0.254071661237785
train_loss= 1.5414981656260305
val_loss= 1.5064195275306702
val_acc= 0.2736156351791531
train_loss= 1.5048561127154858
val_loss= 1.4802341222763062
val_acc= 0.31921824104234525
train_loss= 1.476374414060023
val_loss= 1.4533691465854646
val_acc= 0.30618892508143325
train_loss= 1.4737952235457186
val_loss= 1.4359757065773011
val_acc= 0.36482084690553745
train_loss= 1.4470334486527876
val_loss= 1.4266019880771637
val_acc= 0.3583061889250814
train_loss= 1.434314580706807
val_loss= 1.4120697617530822
val_acc= 0.3355048859934853
train_loss= 1.4293544555639293
val_loss= 1.4124816715717317
val_acc= 0.3322475570032573
train_loss= 1.4235187149667121
val_loss= 1.4078667283058166
val_acc= 0.31921824104234525
train_loss= 1.4155048426095542
val_loss= 1.415875881910324
val_acc= 0.3289902280130293
avg_val 1.44823333799839
avg_train 1.471261151889702


In [36]:
model_finance.load_state_dict(torch.load("best_model_finance.pt"))
model_finance.to(device)
model_finance.eval()
_, _, predictions = evals(X_val_dataloader_finance)
true_labels = []
for batch in X_val_dataloader_finance:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.79      0.33      0.47        33
           1       0.35      0.13      0.19        69
           2       0.22      0.09      0.13        76
           3       0.30      0.89      0.45        80
           4       0.00      0.00      0.00        49

    accuracy                           0.32       307
   macro avg       0.33      0.29      0.25       307
weighted avg       0.30      0.32      0.24       307

MAE 0.9641693811074918


In [37]:
_, _, predictions_train = evals(X_text_dataloader_finance)
true_labels_train = []
for batch in X_text_dataloader_finance:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.12      0.05      0.07       129
           1       0.25      0.11      0.15       276
           2       0.20      0.08      0.12       305
           3       0.25      0.74      0.38       320
           4       0.00      0.00      0.00       196

    accuracy                           0.24      1226
   macro avg       0.17      0.20      0.14      1226
weighted avg       0.19      0.24      0.17      1226

MAE 1.2153344208809136


In [38]:
model_risks = BertClassifier(num_classes=5)
model_risks.to(device)
optimizer_risks = torch.optim.AdamW(model_risks.classifier.parameters(), lr=2e-4)
criterion_risks = nn.CrossEntropyLoss()
model = model_risks
model.to(device)
optimizer = optimizer_risks
criterion = criterion_risks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
X_text_risks = df['Решение кейса'].fillna('').values
y_risks = (df['Анализ рисков'].values - 1).astype("int64")

X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text_risks, y_risks, test_size=0.2, random_state=42, stratify=y_risks
)
X_text_dataset_risks = TextDataset(X_train_text_risks, y_train_risks, tokenizer)
X_val_dataset_risks = TextDataset(X_test_text_risks, y_test_risks, tokenizer)
X_text_dataloader_risks = DataLoader(X_text_dataset_risks, batch_size=16, shuffle=True)
X_val_dataloader_risks = DataLoader(X_val_dataset_risks, batch_size=16, shuffle=False)

In [40]:
best_loss_risks = float("inf")
all_val = 0
all_train = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_risks)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_risks)
  all_val += val_loss
  all_train += train_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_risks:
    best_loss_risks = val_loss
    torch.save(model_risks.state_dict(), "best_model_risks.pt")
print("avg_val=", all_val / 10)
print("avg_train=", all_train/10)

train_loss= 1.5794754926260415
val_loss= 1.537816870212555
val_acc= 0.26384364820846906
train_loss= 1.5521592288822323
val_loss= 1.511128008365631
val_acc= 0.2964169381107492
train_loss= 1.5094680213308953
val_loss= 1.4871526062488556
val_acc= 0.33876221498371334
train_loss= 1.4899914822021088
val_loss= 1.4572062611579895
val_acc= 0.3355048859934853
train_loss= 1.481143646426015
val_loss= 1.4579503059387207
val_acc= 0.32247557003257327
train_loss= 1.4621734092761944
val_loss= 1.430433315038681
val_acc= 0.38436482084690554
train_loss= 1.4465603534277383
val_loss= 1.4201173961162568
val_acc= 0.3713355048859935
train_loss= 1.4286933998008826
val_loss= 1.4053614377975463
val_acc= 0.36156351791530944
train_loss= 1.4196790162619057
val_loss= 1.4060801923274995
val_acc= 0.36156351791530944
train_loss= 1.411917316449153
val_loss= 1.4034157752990724
val_acc= 0.3745928338762215
avg_val= 1.4516662168502807
avg_train= 1.4781261366683167


In [41]:
model_risks.load_state_dict(torch.load("best_model_risks.pt"))
model_risks.to(device)
model_risks.eval()
_, _, predictions = evals(X_val_dataloader_risks)
true_labels = []
for batch in X_val_dataloader_risks:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.76      0.32      0.45        41
           1       0.33      0.42      0.37        64
           2       0.41      0.15      0.22        80
           3       0.35      0.85      0.50        74
           4       0.00      0.00      0.00        48

    accuracy                           0.37       307
   macro avg       0.37      0.35      0.31       307
weighted avg       0.36      0.37      0.31       307

MAE 0.8827361563517915


In [42]:
_, _, predictions_train = evals(X_text_dataloader_risks)
true_labels_train = []
for batch in X_text_dataloader_risks:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

              precision    recall  f1-score   support

           0       0.12      0.06      0.08       166
           1       0.21      0.31      0.25       255
           2       0.24      0.10      0.14       319
           3       0.25      0.54      0.34       297
           4       0.00      0.00      0.00       189

    accuracy                           0.23      1226
   macro avg       0.16      0.20      0.16      1226
weighted avg       0.18      0.23      0.18      1226

MAE 1.2911908646003263


In [ ]:
model_proves = BertClassifier(num_classes=5)
model_proves.to(device)
optimizer_proves = torch.optim.AdamW(model_proves.classifier.parameters(), lr=2e-4)
criterion_proves = nn.CrossEntropyLoss()
model = model_proves
model.to(device)
optimizer = optimizer_proves
criterion = criterion_proves


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
X_text_proves = df['Решение кейса'].fillna('').values
y_proves = (df['Доказательства'].values - 1).astype("int64")

X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text_proves, y_proves, test_size=0.2, random_state=42, stratify=y_proves
)
X_text_dataset_proves = TextDataset(X_train_text_proves, y_train_proves, tokenizer)
X_val_dataset_proves = TextDataset(X_test_text_proves, y_test_proves, tokenizer)
X_text_dataloader_proves = DataLoader(X_text_dataset_proves, batch_size=16, shuffle=True)
X_val_dataloader_proves = DataLoader(X_val_dataset_proves, batch_size=16, shuffle=False)

In [ ]:
best_loss_proves = float("inf")
all_val = 0
all_train = 0
for epoch in range(10):
  train_loss = train(X_text_dataloader_proves)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_proves)
  all_val += val_loss
  all_train += train_loss
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_proves:
    best_loss_proves = val_loss
    torch.save(model_proves.state_dict(), "best_model_proves.pt")
print("avg_val=", all_val/10)
print("avg_train", all_train/10)

train_loss= 1.5791192960739135
val_loss= 1.6087919814246041
val_acc= 0.25
train_loss= 1.5411883354187013
val_loss= 1.6053266354969569
val_acc= 0.25
train_loss= 1.5237300157546998
val_loss= 1.5585596902029855
val_acc= 0.34
train_loss= 1.5101391983032226
val_loss= 1.5365464857646398
val_acc= 0.35
train_loss= 1.5080970001220704
val_loss= 1.524219581059047
val_acc= 0.35
train_loss= 1.45882266998291
val_loss= 1.5270372118268694
val_acc= 0.35
train_loss= 1.4481796503067017
val_loss= 1.501506311552865
val_acc= 0.36
train_loss= 1.4266124153137207
val_loss= 1.4828575168337141
val_acc= 0.36
train_loss= 1.4104590177536012
val_loss= 1.454303468976702
val_acc= 0.36
train_loss= 1.3851968908309937
val_loss= 1.452040774481637
val_acc= 0.36


In [ ]:
model_proves.load_state_dict(torch.load("best_model_proves.pt"))
model_proves.to(device)
model_proves.eval()
_, _, predictions = evals(X_val_dataloader_proves)
true_labels = []
for batch in X_val_dataloader_proves:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.73      0.61      0.67        18
           1       0.00      0.00      0.00        22
           2       0.00      0.00      0.00        23
           3       0.29      1.00      0.45        25
           4       0.00      0.00      0.00        12

    accuracy                           0.36       100
   macro avg       0.21      0.32      0.22       100
weighted avg       0.21      0.36      0.23       100

MAE 0.96


In [ ]:
_, _, predictions_train = evals(X_text_dataloader_proves)
true_labels_train = []
for batch in X_text_dataloader_proves:
  labels = batch["labels"].cpu().numpy()
  true_labels_train.extend(labels)
print(classification_report(true_labels_train, predictions_train))
print("MAE", mean_absolute_error(true_labels_train, predictions_train))

In [ ]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [43]:
def get_prediction_audience(text):
  model_audience.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_audience, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [44]:
def get_prediction_solution(text):
  model_sol.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_sol, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [45]:
def get_prediction_finance(text):
  model_finance.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  model.load_state_dict(torch.load("best_model_audience.pt", map_location=device))
  preds = predicts([text], model_finance, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [46]:
def get_prediction_risks(text):
  model_risks.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_risks, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [47]:
def get_prediction_proves(text):
  model_proves.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_proves, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [48]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
#df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
#df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

In [49]:
df_test_cases['pred_audience']+=1
df_test_cases['pred_finance'] += 1
df_test_cases['pred_risks'] += 1
df_test_cases['pred_sol']+= 1
df_test_cases['pred_proves'] += 1

In [50]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [51]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.3076923076923077
f1micro_audience=  0.3076923076923077
f1macro_audience=  0.2780682839173405
              precision    recall  f1-score   support

           1       0.44      0.17      0.25        23
           2       0.27      0.46      0.34        26
           3       0.37      0.21      0.26        34
           4       0.28      0.68      0.40        22
           5       0.40      0.08      0.13        25

    accuracy                           0.31       130
   macro avg       0.35      0.32      0.28       130
weighted avg       0.35      0.31      0.28       130

MAE= 0.8384615384615385


In [52]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

KeyError: 'pred_sol'

In [53]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.3076923076923077
f1micro_finance=  0.3076923076923077
f1macro_finance=  0.25696969696969696
              precision    recall  f1-score   support

           1       0.60      0.26      0.36        23
           2       0.26      0.21      0.23        33
           3       0.24      0.23      0.23        31
           4       0.31      0.83      0.45        24
           5       0.00      0.00      0.00        19

    accuracy                           0.31       130
   macro avg       0.28      0.31      0.26       130
weighted avg       0.29      0.31      0.26       130

MAE= 0.9846153846153847


In [54]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.3153846153846154
f1micro_risks=  0.3153846153846154
f1macro_risks=  0.26178839957035444
              precision    recall  f1-score   support

           1       0.44      0.15      0.23        26
           2       0.30      0.41      0.34        32
           3       0.21      0.13      0.16        30
           4       0.36      0.70      0.47        27
           5       0.20      0.07      0.10        15

    accuracy                           0.32       130
   macro avg       0.30      0.29      0.26       130
weighted avg       0.31      0.32      0.28       130

MAE= 0.9384615384615385


In [ ]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.14615384615384616
f1micro_proves=  0.14615384615384616
f1macro_proves=  0.07767722473604827
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        18
           2       0.00      0.00      0.00        38
           3       0.17      0.40      0.24        25
           4       0.15      0.47      0.23        19
           5       0.00      0.00      0.00        30

    accuracy                           0.15       130
   macro avg       0.05      0.15      0.08       130
weighted avg       0.05      0.15      0.08       130

MAE 1.176923076923077


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [ ]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [ ]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.9538461538461539


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
